In [ ]:
import os
os.chdir(os.environ['PWD'])

import numpy as np
import yaml
from utilities.utils import correct_type_of_entry, get_exp_file_name, create_all_configs, get_min_max_loss
from bounds.kl_inv import kl_inv
import json
import pandas as pd
from itertools import product
import os

In [ ]:
def slicen(n):
    return (slice(None),)*n


zero_one_loss = False

if zero_one_loss:
    choosing_parameter = "full_disagreement"
    # choosing_parameter = "validation_error"
    values_to_fetch = ['complement_error', 'validation_error', 'test_error', 'best_model_complement_error','best_model_test_error',
                       'kl_bound', 'disagreement_bound', 'full_disagreement', 'kl_val']
else:
    choosing_parameter = "full_disagreement_loss"
    # choosing_parameter = "validation_loss"
    values_to_fetch = ['complement_loss', 'validation_loss', 'test_loss', 'best_model_complement_loss', 'best_model_test_loss','CE_kl_bound',
                       'disagreement_loss_kl', 'full_disagreement_loss', 'kl_val']
    

In [ ]:
results_list = []
dataset_list = []

for dataset in ['mnist', 'cifar10']:
    dataset_config_name = "./configs/dataset_configs/" + dataset + ".yaml"
    with open(dataset_config_name) as file:
        dataset_configuration = yaml.safe_load(file)
    
    sweep_config_name = "./configs/experiment_configs/pbb/" + dataset + ".yaml"
    with open(sweep_config_name) as file:
        sweep_configuration = yaml.safe_load(file)
    
    hps = {}
    for key, item in sweep_configuration['parameters'].items():
        if item.get('values', None) is not None:
            hps[key] = correct_type_of_entry(item['values'])
    size_hyperparams = tuple([len(l) for l in hps.values()])
    
    results_matrix = np.ones(((len(values_to_fetch),) + size_hyperparams))
    results_matrix[:] = np.nan
    
    for sweep_config in create_all_configs(sweep_configuration):
        file_name = get_exp_file_name(sweep_config|dataset_configuration, path="./pbb_logs/")
        if not os.path.exists(file_name):
            print(file_name)
        if os.path.exists(file_name):
            with open(file_name) as f:
                config = json.load(f)
                for val_to_fetch_idx in range(len(values_to_fetch)):
                    matrix_idx = tuple([val_to_fetch_idx] + [hps[key].index(config['config'].get(key, None)) for key in hps.keys()])
                    val_to_fetch = values_to_fetch[val_to_fetch_idx]
                    if val_to_fetch == 'full_disagreement':
                        disag_ = config['disagreement']
                        first_bound = kl_inv(disag_, -np.log(config['config']['delta']/2)/config['config']['mc_samples'], 'MAX')
                        n_data_disag = config['validation_set_size'] / config['config']['validation_size'] * (1-config['config']['validation_size']) * config['config']['disagreement_size']
                        
                        second_bound = kl_inv(first_bound, -np.log(config['config']['delta']/2)/n_data_disag , 'MAX')
                        if second_bound - config['disagreement_bound'] < 0:
                            print("HEY")

                        new_full = config['kl_bound'] + second_bound
                        if new_full - config['full_disagreement'] < 0 :
                            print("WHAT")
                        results_matrix[matrix_idx] = new_full
                    elif val_to_fetch == 'full_disagreement_loss':
                        min_val, max_val = get_min_max_loss(config['config']['min_probability'],
                                         config['config']['n_classes'],
                                         config['config']['clamp_method'])
                        disag_ = config['loss_disagreement'] / (max_val - min_val)
                        first_bound = kl_inv(disag_, -np.log(config['config']['delta']/2)/config['config']['mc_samples'], 'MAX')
                        n_data_disag = config['validation_set_size'] / config['config']['validation_size'] * (1-config['config']['validation_size']) * config['config']['disagreement_size']
                        second_bound = (max_val - min_val) * kl_inv(first_bound, -np.log(config['config']['delta']/2)/n_data_disag , 'MAX')
                        print("Second bound : ", second_bound, 'original bound : ', config['disagreement_loss_kl'])
                        if second_bound - config['disagreement_loss_kl'] < 0 :
                            print("HEY")
                        new_full = config['CE_kl_bound'] + second_bound
                        if new_full - config['full_disagreement'] < 0 :
                            print("WHAT")
                        results_matrix[matrix_idx] = new_full
                    else:
                        results_matrix[matrix_idx] = config.get(val_to_fetch, None)
    
    hp_list = list(hps.values())[1:]
    params_product = list(product(*hp_list))
    name_list = []
    idx_list = []
    for params in params_product:
        name = ""
        for p in params:
            name += str(p) + " "
        name_list.append(name[:-1])
        idx = ()
        for p_idx in range(len(params)):
            p_key = list(hps.keys())[1:][p_idx]
            idx += (hps[p_key].index(params[p_idx]),)
        idx_list.append(tuple(idx))
    
    reshaped_matrix = np.mean(results_matrix, axis=1).reshape(results_matrix.shape[0],np.prod(results_matrix.shape[2:])).T
    mean_df = pd.DataFrame(reshaped_matrix, index=name_list, columns=values_to_fetch)
    
    reshaped_matrix_std = np.nanstd(results_matrix, axis=1).reshape(results_matrix.shape[0],np.prod(results_matrix.shape[2:])).T
    std_df = pd.DataFrame(reshaped_matrix_std, index=name_list, columns=values_to_fetch)
    
    if "model_type" in list(hps.keys()):
        list_indexs = [[] for _ in range(len(hps['model_type']))]
        for name in list(mean_df.index):
            for model_idx in range(len(hps['model_type'])):
                if hps['model_type'][model_idx] in name:
                    list_indexs[model_idx].append(name)
                    break
    else:
        list_indexs = [list(mean_df.index)]
    
    for i in range(len(list_indexs)):
        index_mean = mean_df.loc[list_indexs[i]].sort_values(by=choosing_parameter).index[0]
        
        best_datapoint_mean = mean_df.loc[index_mean]
        best_datapoint_std = std_df.loc[index_mean]

        
        if "model_type" in list(hps.keys()):
            if hps['model_type'][i] == "cnn":
                mt = "CNN"
            elif hps['model_type'][i] == "resnet":
                mt = "ResNet18"
            else:
                raise NotImplementedError
        else:
            if dataset == "mnist":
                mt = "CNN"
            elif dataset == "cifar10":
                mt = "ResNet18"
            else:
                raise NotImplementedError("Error in the model type selection")
    
        rounding_val = 2 if zero_one_loss else 4
        rounding = f"%.{rounding_val}f"
        
        mult = 100.0 if zero_one_loss else 1.0
    
        if zero_one_loss:
            vals_to_add = ["Model", 'complement_error', 'test_error', 'kl_val', 'best_model_complement_error', 'best_model_test_error',
                            'kl_bound', 'disagreement_bound', 'full_disagreement']
        else:
            vals_to_add = ["Model", 'complement_loss', 'test_loss', 'kl_val', 'best_model_complement_loss',
                           'best_model_test_loss', 'CE_kl_bound', 'disagreement_loss_kl', 'full_disagreement_loss']
    
        list_of_vals = []
        for val in vals_to_add:
            if val == "Model":
                list_of_vals.append(mt)
                continue
            elif val == 'kl_val':
                temp = (rounding % round(best_datapoint_mean[val],rounding_val)) + r"$\pm$"
                temp += (rounding % round(best_datapoint_std[val], rounding_val))
                list_of_vals.append(temp)
                continue
    
            temp = (rounding % round(mult*best_datapoint_mean[val],rounding_val)) + r"$\pm$"
            temp += (rounding % round(mult*best_datapoint_std[val], rounding_val))
            list_of_vals.append(temp)

        results_list.append(pd.Series(list_of_vals, index=vals_to_add))
            
        dataset_list.append(dataset.upper())

if zero_one_loss:
    print(pd.DataFrame(results_list, index=dataset_list).to_latex(float_format="%.2f"))
else:
    print(pd.DataFrame(results_list, index=dataset_list).to_latex(float_format="%.4f"))
    